# Metis YOLOv8 Logo Detector Training
This notebook is designed to run on Google Colab (with a GPU) to train the YOLOv8 model on the Brand Eye Dataset for 50-100 epochs to achieve production-level accuracy (>90% mAP).

In [ ]:
!pip install ultralytics huggingface_hub

## 1. Download Dataset

In [ ]:
import os
from huggingface_hub import snapshot_download

print("Downloading logo dataset from Hugging Face...")
local_dir = "/content/data/logo_dataset"
os.makedirs(local_dir, exist_ok=True)

snapshot_download(
    repo_id="haydarkadioglu/brand-eye-dataset",
    repo_type="dataset",
    local_dir=local_dir,
    max_workers=8
)
print(f"Dataset successfully downloaded to {local_dir}")

## 2. Prepare `data.yaml`
We need to update the dataset's configuration file to use absolute paths in the Colab environment.

In [ ]:
import yaml

yaml_path = "/content/data/logo_dataset/data.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = "/content/data/logo_dataset"
data['train'] = "train/images"
data['val'] = "valid/images"
data['test'] = "test/images"

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)
    
print("data.yaml updated successfully for Colab!")

## 3. Train YOLOv8 Model
Run the training for 50 epochs using the free T4 GPU. This should take 15-30 minutes.

In [ ]:
from ultralytics import YOLO

# Load pre-trained model
model = YOLO("yolov8n.pt")

# Train the model on the T4 GPU (device=0)
results = model.train(
    data="/content/data/logo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/ml_models",
    name="logo_detection"
)

## 4. Download the Trained Weights
Run this cell to download the `best.pt` file to your local computer. Once downloaded, place it in your project at `runs/detect/ml/models/logo_detection/weights/best.pt` (or adjust the path in your `config.py`).

In [ ]:
from google.colab import files

best_weights_path = "/content/ml_models/logo_detection/weights/best.pt"
files.download(best_weights_path)